In [ ]:
import pandas as  pd
import matplotlib.pyplot as plt
movies_df = pd.read_csv('../data/movies.csv')
ratings_df = pd.read_csv('../data/ratings.csv')


In [2]:
movie_stats = ratings_df.groupby('movieId')['rating'].agg(['count', 'mean']).reset_index()
movie_stats.columns = ['movieId', 'rating_count', 'rating_mean']
print(movie_stats.head())

   movieId  rating_count  rating_mean
0        1           215     3.920930
1        2           110     3.431818
2        3            52     3.259615
3        4             7     2.357143
4        5            49     3.071429


In [3]:
movie_stats_filtered = movie_stats[movie_stats['rating_count'] >= 10]
print(movie_stats_filtered.head())

   movieId  rating_count  rating_mean
0        1           215     3.920930
1        2           110     3.431818
2        3            52     3.259615
4        5            49     3.071429
5        6           102     3.946078


In [4]:
df_final = pd.merge(movies_df, movie_stats_filtered, on= 'movieId', how='inner')
df_final.head()


,movieId,title,genres,rating_count,rating_mean
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,215,3.920930
1,2,Jumanji (1995),Adventure|Children|Fantasy,110,3.431818
2,3,Grumpier Old Men (1995),Comedy|Romance,52,3.259615
3,5,Father of the Bride Part II (1995),Comedy,49,3.071429
4,6,Heat (1995),Action|Crime|Thriller,102,3.946078


In [5]:
print(f"Nombre de films après filtrage: {len(df_final)}")
df_final.sort_values(by='rating_mean', ascending=False).head()

Nombre de films après filtrage: 2269


,movieId,title,genres,rating_count,rating_mean
411,1041,Secrets & Lies (1996),Drama,11,4.590909
1186,3451,Guess Who's Coming to Dinner (1967),Drama,11,4.545455
456,1178,Paths of Glory (1957),Drama|War,12,4.541667
441,1104,"Streetcar Named Desire, A (1951)",Drama,20,4.475000
871,2360,"Celebration, The (Festen) (1998)",Drama,12,4.458333


In [6]:
genre_list = df_final['genres'].str.split('|')


In [7]:
from sklearn.preprocessing import MultiLabelBinarizer
mlb = MultiLabelBinarizer()
genres_matrix = mlb.fit_transform(genre_list)
genres_df = pd.DataFrame(genres_matrix, columns=mlb.classes_, index = df_final.index)
df_final = pd.concat([df_final, genres_df], axis=1)
df_final.head()

,movieId,title,genres,rating_count,rating_mean,Action,Adventure,Animation,Children,Comedy,...,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,215,3.920930,0,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
1,2,Jumanji (1995),Adventure|Children|Fantasy,110,3.431818,0,1,0,1,0,...,0,0,0,0,0,0,0,0,0,0
2,3,Grumpier Old Men (1995),Comedy|Romance,52,3.259615,0,0,0,0,1,...,0,0,0,0,0,1,0,0,0,0
3,5,Father of the Bride Part II (1995),Comedy,49,3.071429,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
4,6,Heat (1995),Action|Crime|Thriller,102,3.946078,1,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0


In [8]:
from sklearn.metrics.pairwise import cosine_similarity
cosine_sim = cosine_similarity(genres_matrix)

sim_df = pd.DataFrame(cosine_sim, index=df_final['title'], columns=df_final['title'])
sim_df.head()

title,Toy Story (1995),Jumanji (1995),Grumpier Old Men (1995),Father of the Bride Part II (1995),Heat (1995),Sabrina (1995),Sudden Death (1995),GoldenEye (1995),"American President, The (1995)",Dracula: Dead and Loving It (1995),...,Moana (2016),Rogue One: A Star Wars Story (2016),Hidden Figures (2016),Get Out (2017),Logan (2017),Dunkirk (2017),Blade Runner 2049 (2017),Coco (2017),Star Wars: The Last Jedi (2017),Deadpool 2 (2018)
title,,,,,,,,,,,,,,,,,,,,,
Toy Story (1995),1.000000,0.774597,0.316228,0.447214,0.0,0.316228,0.00000,0.258199,0.258199,0.316228,...,1.000000,0.447214,0.0,0.0,0.000000,0.00000,0.0,0.774597,0.447214,0.258199
Jumanji (1995),0.774597,1.000000,0.000000,0.000000,0.0,0.000000,0.00000,0.333333,0.000000,0.000000,...,0.774597,0.577350,0.0,0.0,0.000000,0.00000,0.0,0.666667,0.577350,0.000000
Grumpier Old Men (1995),0.316228,0.000000,1.000000,0.707107,0.0,1.000000,0.00000,0.000000,0.816497,0.500000,...,0.316228,0.000000,0.0,0.0,0.000000,0.00000,0.0,0.000000,0.000000,0.408248
Father of the Bride Part II (1995),0.447214,0.000000,0.707107,1.000000,0.0,0.707107,0.00000,0.000000,0.577350,0.707107,...,0.447214,0.000000,0.0,0.0,0.000000,0.00000,0.0,0.000000,0.000000,0.577350
Heat (1995),0.000000,0.000000,0.000000,0.000000,1.0,0.000000,0.57735,0.666667,0.000000,0.000000,...,0.000000,0.288675,0.0,0.0,0.408248,0.57735,0.0,0.000000,0.288675,0.333333


In [9]:
df_final = df_final[df_final['genres'] != '(no genres listed)']

In [ ]:
def recommend_movies(movie_title, n = 5):
    matches = df_final[df_final['title'].str.contains(movie_title, case=False, na=False)]
    if matches.empty:
        return f"Désolé, aucun film trouvé pour '{movie_title}'."
    
    ref_movie_title = matches.iloc[0]['title']
    print(f"Recherche de recommandations pour : {ref_movie_title}")
    sim_scores = sim_df[ref_movie_title].copy()
    
    rec_df = pd.DataFrame(sim_scores).rename(columns={ref_movie_title: 'similarity'})
    rec_df = rec_df.merge(df_final[['title', 'rating_mean', 'rating_count', 'genres']], on='title')
    rec_df = rec_df[rec_df['title'] != ref_movie_title]

    recommendations = rec_df.sort_values(by=['similarity', 'rating_mean', 'rating_count'], ascending=[False, False, False])
    return recommendations.head(n)

In [11]:
# Test de la fonction
resultats = recommend_movies("Matrix", n=5)
print(resultats[['title', 'similarity', 'rating_count', 'rating_mean']])


Recherche de recommandations pour : Matrix, The (1999)
                                  title  similarity  rating_count  rating_mean
261                 Blade Runner (1982)         1.0           124     4.100806
501              Terminator, The (1984)         1.0           131     3.896947
1522                 Equilibrium (2002)         1.0            44     3.875000
954                     eXistenZ (1999)         1.0            22     3.863636
2227  Captain America: Civil War (2016)         1.0            22     3.613636
